In [4]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/RAG-Chat/data/medical_test_350_expanded.parquet")

print(df.shape)  # 행·열 개수
print(df.columns.tolist())
print(df[["comparison_group_id", "query_variant_type", "source_query", "query"]].head(12))

Mounted at /content/drive
(1400, 21)
['query_id', 'split', 'query', 'query_type', 'answers', 'pool', 'context', 'label', 'high_risk', 'time_sensitive', 'parse_ok', 'raw', 'prompt_version', 'gen_model', 'question_category', 'matched_keyword', 'source_query_id', 'comparison_group_id', 'source_query', 'query_variant_type', 'requires_paired_context']
    comparison_group_id query_variant_type  \
0                 29704                 원본   
1                 29704                구어체   
2                 29704               쉬운 말   
3                 29704           핵심 단어 회피   
4                 88293                 원본   
5                 88293                구어체   
6                 88293               쉬운 말   
7                 88293           핵심 단어 회피   
8                 90805                 원본   
9                 90805                구어체   
10                90805               쉬운 말   
11                90805           핵심 단어 회피   

                                        source_query

In [5]:
pd.set_option("display.max_colwidth", None)

# 원본 질문 하나에 딸린 변형 질문 4개를 전체 문장으로 보기
g = df[df.comparison_group_id == 29704]
display(g[["query_variant_type", "query", "answers"]])
print(g.context.iloc[0])  # 4개 변형이 같이 쓰는 지문

# 분포 확인
display(df.query_variant_type.value_counts())
display(df.question_category.value_counts())


,query_variant_type,query,answers
0,원본,what does antiviral medication do,"[Treating viral infections., They help fight flu, according to the Centers for Disease Control and Prevention.]"
1,구어체,Quick question: what does antiviral medication do?,"[Treating viral infections., They help fight flu, according to the Centers for Disease Control and Prevention.]"
2,쉬운 말,"In simple words, what does antiviral medicine do?","[Treating viral infections., They help fight flu, according to the Centers for Disease Control and Prevention.]"
3,핵심 단어 회피,"From the passage, what does this medicine do?","[Treating viral infections., They help fight flu, according to the Centers for Disease Control and Prevention.]"


Antivirals are medications that help fight flu, according to the Centers for Disease Control and Prevention. They're available only with a prescription from a health care professional. Antiviral drugs work against viruses; they don't fight bacteria like antibiotics do. The two antivirals used as flu treatments are oseltamivir phosphate (Tamiflu) and zanamivir (Relenza).


,count
query_variant_type,
원본,350
구어체,350
쉬운 말,350
핵심 단어 회피,350


,count
question_category,
정의·의미,332
원인,188
해부·생리,136
검사·정상수치,124
기타,112
기간·시기,76
효능·용도,68
약물,60
증상,56


In [28]:
from google.colab import data_table

# 전체 행을 정렬·검색·페이지 넘김이 되는 표로 보기
view = df[["comparison_group_id", "query_variant_type", "query", "answers", "context",
           "label", "question_category", "requires_paired_context"]].copy()
view["answers"] = view.answers.map(" | ".join)  # 배열은 표에 안 나와서 문자열로
data_table.DataTable(view, num_rows_per_page=20)

,comparison_group_id,query_variant_type,query,answers,context,label,question_category,requires_paired_context
0,29704,원본,what does antiviral medication do,Treating viral infections. | They help fight f...,Antivirals are medications that help fight flu...,A,약물,False
1,29704,구어체,Quick question: what does antiviral medication...,Treating viral infections. | They help fight f...,Antivirals are medications that help fight flu...,A,약물,False
2,29704,쉬운 말,"In simple words, what does antiviral medicine do?",Treating viral infections. | They help fight f...,Antivirals are medications that help fight flu...,A,약물,False
3,29704,핵심 단어 회피,"From the passage, what does this medicine do?",Treating viral infections. | They help fight f...,Antivirals are medications that help fight flu...,A,약물,True
4,88293,원본,what is bun mean in blood work,A blood urea nitrogen (BUN) test measures the ...,A blood urea nitrogen (BUN) test measures the ...,A,정의·의미,False
...,...,...,...,...,...,...,...,...
1395,42181,핵심 단어 회피,"From the passage, what can cause this problem?",African sleeping sickness is a disease caused ...,"Symptoms include mucusy stools, diarrhea, naus...",A,원인,True
1396,48874,원본,aspirin or advil for arthritis,Aspirin,Uses. Aspirin is used to reduce fever and reli...,A,기타,False
1397,48874,구어체,I was wondering: aspirin or advil for arthritis?,Aspirin,Uses. Aspirin is used to reduce fever and reli...,A,기타,False
1398,48874,쉬운 말,"In simple words, aspirin or advil for arthritis?",Aspirin,Uses. Aspirin is used to reduce fever and reli...,A,기타,False
